# Classification NBA Model

## Configuration

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training
)
from sklearn.model_selection import cross_validate, train_test_split
from xgboost import XGBClassifier



## Load Data

In [2]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260227.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)


In [3]:
# #print column names with name in lower case
# for col in df_stats.columns:
#     if not col.isupper():
#         print(col)

In [4]:
#filter for last two seasons, gett seasons 2025 and 2024
# df_stats = df_stats[df_stats['SEASON_YEAR'].isin([2024, 2025])]

In [5]:
df_stats

,TOTAL_OVER_UNDER_LINE,TEAM_ID_TEAM_HOME,TEAM_CITY_TEAM_HOME,TEAM_ABBREVIATION_TEAM_HOME,TEAM_NAME_TEAM_HOME,MATCHUP_TEAM_HOME,GAME_NUMBER_TEAM_HOME,TEAM_ID_TEAM_AWAY,TEAM_CITY_TEAM_AWAY,TEAM_ABBREVIATION_TEAM_AWAY,...,TRAVEL_RECENCY_RATIO_AWAY_2D_OVER_14D,REST_DAYS_DIFF_HOME_MINUS_AWAY,INJURY_PTS_SHARE_HOME,INJURY_PTS_SHARE_AWAY,STAR_PTS_PCT_DIFF_HOME_MINUS_AWAY,POSS_X_TSPCT_HOME,POSS_X_TSPCT_AWAY,IS_WEEKEND,MONTH,IS_US_HOLIDAY
0,228.5,1610612749,Milwaukee,MIL,Milwaukee Bucks,MIL vs. CLE,57,1610612739,Cleveland,CLE,...,0.847695,0,0.325611,0.571340,0.023316,59.498649,60.019467,0,2,0
1,226.5,1610612763,Memphis,MEM,Memphis Grizzlies,MEM vs. GSW,57,1610612744,Golden State,GSW,...,1.000000,1,0.789192,0.930473,0.042215,54.826883,58.091218,0,2,0
2,221.5,1610612745,Houston,HOU,Houston Rockets,HOU vs. SAC,57,1610612758,Sacramento,SAC,...,0.788499,0,0.215040,0.495445,0.060424,55.575318,51.381648,0,2,0
3,218.5,1610612765,Detroit,DET,Detroit Pistons,DET vs. OKC,57,1610612760,Oklahoma City,OKC,...,0.911382,1,0.063098,0.784668,0.099422,53.835189,61.279633,0,2,0
4,228.5,1610612761,Toronto,TOR,Toronto Raptors,TOR vs. SAS,59,1610612759,San Antonio,SAS,...,0.893297,-1,0.087929,0.057939,-0.019391,55.958915,57.830834,0,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11092,NaN,1610612759,San Antonio,SAS,San Antonio Spurs,SAS vs. MIN,1,1610612750,Minnesota,MIN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0
11093,NaN,1610612763,Memphis,MEM,Memphis Grizzlies,MEM vs. NOP,1,1610612740,New Orleans,NOP,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0
11094,NaN,1610612754,Indiana,IND,Indiana Pacers,IND vs. BKN,1,1610612751,Brooklyn,BKN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0
11095,NaN,1610612739,Cleveland,CLE,Cleveland Cavaliers,CLE vs. BOS,1,1610612738,Boston,BOS,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0


In [6]:

df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=5, drop_all_na_rows=True, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'])


STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11097 rows
Basic cleaning complete: 8490 rows remaining

Starting advanced column cleaning with 1779 columns

Advanced column cleaning complete: 1779 → 963 columns (816 removed)


Dropping NA rows for SEASON_YEAR 2017...
   Removed 0 rows with NaN values from 2017 season

Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 11 (0.13%)
  Critical columns requiring data: 5
  Columns zero-filled: 210
  Infer pairs applied: 8/248
  Remaining NaN cells: 27016

Dropping rows that contain NaN...
CLEANING COMPLETE
Final shape: (7297, 963)


In [7]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_stats[df_stats[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_stats.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year


In [8]:
BET365_LINE_COL =  "total_bet365_line_over"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

In [9]:
print(f" Total Seasons in dataset: {df_to_train['SEASON_YEAR'].nunique()}")
print(f" Total Games in dataset: {len(df_to_train)}")
print(f"Seasons: {df_to_train['SEASON_YEAR'].unique()}")
#print number of games per season
print(df_to_train['SEASON_YEAR'].value_counts())

 Total Seasons in dataset: 7
 Total Games in dataset: 7297
Seasons: [2019 2020 2021 2022 2023 2024 2025]
SEASON_YEAR
2021    1289
2024    1288
2022    1281
2023    1182
2020    1100
2025     833
2019     324
Name: count, dtype: int64


## Train / Test

In [10]:
X = df_to_train.drop(["TOTAL_POINTS", "SEASON_YEAR", "GAME_DATE"], axis=1, errors="ignore")
y = pd.to_numeric(df_to_train["TOTAL_POINTS"], errors="coerce")

In [11]:
# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=16)

In [12]:
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
# Check number of coulmns
print(f"Number of columns in training set: {X_train.shape[1]}")
print(f"Number of columns in test set: {X_test.shape[1]}")

Training set size: 6202
Test set size: 1095
Number of columns in training set: 960
Number of columns in test set: 960


## Cross-validation

In [13]:
from sklearn.model_selection import TimeSeriesSplit, cross_validate, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer, root_mean_squared_error


In [14]:
# Declare time-series cross-validation strategy
kf = TimeSeriesSplit(n_splits=5)


In [15]:
import numpy as np


def over_under_betting_accuracy(
    true_total: np.ndarray,
    pred_total: np.ndarray,
    betting_line: np.ndarray,
) -> float:
    true_total = np.asarray(true_total, dtype=float)
    pred_total = np.asarray(pred_total, dtype=float)
    betting_line = np.asarray(betting_line, dtype=float)

    valid = np.isfinite(true_total) & np.isfinite(pred_total) & np.isfinite(betting_line)
    if not np.any(valid):
        return np.nan

    true_total = true_total[valid]
    pred_total = pred_total[valid]
    betting_line = betting_line[valid]

    true_side = np.where(
        true_total > betting_line,
        "OVER",
        np.where(true_total < betting_line, "UNDER", "PUSH"),
    )
    pred_side = np.where(
        pred_total > betting_line,
        "OVER",
        np.where(pred_total < betting_line, "UNDER", "PUSH"),
    )

    non_push = (true_side != "PUSH") & (pred_side != "PUSH")
    if not np.any(non_push):
        return np.nan

    return float(np.mean(true_side[non_push] == pred_side[non_push]))


class OverUnderScorer:
    """Custom sklearn-compatible scorer for over/under betting accuracy using bet365 line."""

    def __init__(self, line_col: str):
        self.line_col = line_col

    def __call__(self, estimator, X, y_true):
        if self.line_col not in X.columns:
            raise KeyError(f"{self.line_col} not found in X for scoring")

        y_pred = estimator.predict(X)
        betting_line = pd.to_numeric(X[self.line_col], errors="coerce").to_numpy(dtype=float)

        return over_under_betting_accuracy(
            true_total=np.asarray(y_true, dtype=float),
            pred_total=np.asarray(y_pred, dtype=float),
            betting_line=betting_line,
        )


over_under_scorer = OverUnderScorer(BET365_LINE_COL)


In [16]:
# Declare scores to be used
scoring = {
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(root_mean_squared_error),
    'MAE': make_scorer(mean_absolute_error),
    'OU_Betting_Accuracy': over_under_scorer,
}

In [17]:
def print_metrics(cv_results):
    for sc in scoring.keys():
        if sc == 'OU_Betting_Accuracy':
            print(f'Train {sc}:', f"{cv_results[f'train_{sc}'].mean():.2%}")
            print(f'Validation {sc}:', f"{cv_results[f'test_{sc}'].mean():.2%}")
        else:
            print(f'Train {sc}:', cv_results[f'train_{sc}'].mean().round(5))
            print(f'Validation {sc}:', cv_results[f'test_{sc}'].mean().round(5))
        print()

In [18]:
def real_vs_pred(model, X_train, y_train):
    preds = cross_val_predict(model, X_train, y_train, cv=kf, n_jobs=-1)
    x_line = np.arange(y_train.min(), y_train.max())
    plt.scatter(y_train, preds)
    plt.plot(x_line, x_line, color='orange')
    plt.xlabel('Real target')
    plt.ylabel('Predicted target')
    plt.show()

## Baseline

In [19]:
from sklearn.dummy import DummyRegressor

season_bl = DummyRegressor(strategy='mean')
cv_results = cross_validate(season_bl, X_train, y_train, cv=kf,
                            scoring=scoring, return_train_score=True)
season_bl.fit(X_train, y_train)
print_metrics(cv_results)

Train MSE: 369.2664
Validation MSE: 392.42962

Train RMSE: 19.21228
Validation RMSE: 19.80511

Train MAE: 15.34328
Validation MAE: 15.81192

Train OU_Betting_Accuracy: 50.06%
Validation OU_Betting_Accuracy: 50.22%



In [20]:
# Baseline 3: Predict the bet365 betting line as total points
if BET365_LINE_COL not in X_train.columns:
    raise KeyError(f"{BET365_LINE_COL} not found in X_train")

y_pred_baseline_3 = pd.to_numeric(X_train[BET365_LINE_COL], errors="coerce")
valid = y_pred_baseline_3.notna() & pd.to_numeric(y_train, errors="coerce").notna()

# Evaluate
mse = mean_squared_error(y_train[valid], y_pred_baseline_3[valid])
mae = mean_absolute_error(y_train[valid], y_pred_baseline_3[valid])
rmse = root_mean_squared_error(y_train[valid], y_pred_baseline_3[valid])
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")


MSE: 297.40
RMSE: 17.25
MAE: 13.75


## Logistic Regression

In [21]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
cv_results = cross_validate(lr, X_train.fillna(0), y_train, cv=kf,
                            scoring=scoring, return_train_score=True)

print_metrics(cv_results)

Train MSE: 166.68041
Validation MSE: 3911434.95342

Train RMSE: 12.35356
Validation RMSE: 902.20831

Train MAE: 9.77578
Validation MAE: 50.2799

Train OU_Betting_Accuracy: 73.23%
Validation OU_Betting_Accuracy: 51.94%



In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

# Example XGBoost regressor:
xgb_reg = XGBRegressor(
    max_depth=4,
    learning_rate=0.02,
    n_estimators=350,
    subsample=0.5,       # Equivalent to max_samples in GBRegressor
    colsample_bytree=0.6, # Equivalent to max_features in GBRegressor
    n_jobs=-2,
    random_state=16)

cv_results = cross_validate(
    xgb_reg, 
    X_train, y_train, 
    cv=kf, 
    scoring=scoring,      # Use your custom scoring or e.g. 'neg_mean_absolute_error'
    return_train_score=True,
    n_jobs=-2
)
# Train final model on full train set
xgb_reg.fit(X_train, y_train)

# Print metrics
print_metrics(cv_results)


Train MSE: 139.42478
Validation MSE: 301.87411

Train RMSE: 11.59229
Validation RMSE: 17.36982

Train MAE: 9.23009
Validation MAE: 13.80596

Train OU_Betting_Accuracy: 84.94%
Validation OU_Betting_Accuracy: 53.15%



In [23]:
feature_importances = xgb_reg.feature_importances_
important_features = np.argsort(feature_importances)[::-1]  
feature_importances = pd.DataFrame({
    'Feature': X_train.columns[important_features],
    'Importance': feature_importances[important_features]
}).sort_values(by="Importance", ascending=False)
feature_importances



,Feature,Importance
0,total_fanduel_line_over,0.009755
1,total_draftkings_line_over,0.008959
2,odds_total_line_books_mean,0.007821
3,total_caesars_line_over,0.006456
4,odds_implied_points_max,0.006222
...,...,...
939,IS_HOME_WEST_CONFERENCE,0.000000
940,odds_spread_home_is_fav_fanduel,0.000000
941,TOP4_INJURED_STREAK_PACE_PER40_BEFORE_TEAM_AWAY,0.000000
942,IS_OVER_LINE_LAST_ALL_5_MATCHES_BEFORE_TEAM_HOME,0.000000


In [25]:
import numpy as np
import pandas as pd


def evaluate_ou_thresholds(
    model,
    X_test: pd.DataFrame,
    y_test_total: pd.Series,
    line_col: str,
    thresholds=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10),
):
    """Evaluate OU accuracy at different confidence thresholds using bet365 line."""
    if line_col not in X_test.columns:
        raise KeyError(f"{line_col} not found in X_test")

    y_pred_total = np.asarray(model.predict(X_test), dtype=float)
    y_true_total = pd.to_numeric(y_test_total, errors="coerce").to_numpy(dtype=float)
    line = pd.to_numeric(X_test[line_col], errors="coerce").to_numpy(dtype=float)

    margin = np.abs(y_pred_total - line)

    rows = []
    n_total = len(y_true_total)

    for t in thresholds:
        mask = margin > t
        n = int(mask.sum())

        if n == 0:
            acc = np.nan
        else:
            acc = over_under_betting_accuracy(
                true_total=y_true_total[mask],
                pred_total=y_pred_total[mask],
                betting_line=line[mask],
            )

        rows.append(
            {
                "threshold_abs_pred_vs_line_gt": t,
                "n_games": n,
                "pct_of_test": (n / n_total) if n_total else np.nan,
                "ou_accuracy": acc,
            }
        )

    return pd.DataFrame(rows), y_pred_total


results_df, y_pred_test_total = evaluate_ou_thresholds(
    model=xgb_reg,
    X_test=X_test,
    y_test_total=y_test,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_accuracy": "{:.2%}"}
    )
)


,threshold_abs_pred_vs_line_gt,n_games,pct_of_test,ou_accuracy
0,0,1095,100.0%,54.96%
1,1,805,73.5%,54.91%
2,2,529,48.3%,56.26%
3,3,347,31.7%,56.14%
4,4,208,19.0%,60.29%
5,5,131,12.0%,61.07%
6,6,88,8.0%,67.05%
7,7,61,5.6%,70.49%
8,8,41,3.7%,78.05%
9,9,31,2.8%,90.32%


# Optuna

In [26]:
import numpy as np
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit

# Use forward-chaining folds for time-ordered validation
kf = TimeSeriesSplit(n_splits=5)


def ou_accuracy_with_threshold_totals(
    true_total: np.ndarray,
    pred_total: np.ndarray,
    betting_line: np.ndarray,
    threshold_abs: float = 0.0,
    min_coverage: float = 0.25,
):
    # "Edge" in points relative to the line
    margin = np.abs(pred_total - betting_line)

    # Bet only when predicted edge exceeds threshold
    mask = margin > threshold_abs
    coverage = float(np.mean(mask))

    if coverage < min_coverage or mask.sum() == 0:
        return 0.0, coverage

    score = over_under_betting_accuracy(
        true_total=true_total[mask],
        pred_total=pred_total[mask],
        betting_line=betting_line[mask],
    )

    # If all filtered as pushes/no-valid, penalize
    if np.isnan(score):
        return 0.0, coverage

    return score, coverage


def _predict_best(model, X):
    # Use best iteration if early stopping happened
    if getattr(model, "best_iteration", None) is not None:
        # Newer XGBoost
        try:
            return model.predict(X, iteration_range=(0, model.best_iteration + 1))
        except TypeError:
            # Older compatibility path
            ntree_limit = getattr(model, "best_ntree_limit", None)
            if ntree_limit is not None:
                return model.predict(X, ntree_limit=ntree_limit)
    return model.predict(X)


def objective(trial, X, y):
    if BET365_LINE_COL not in X.columns:
        raise KeyError(f"{BET365_LINE_COL} not found in X")

    threshold_abs = trial.suggest_float("threshold_abs", 0.0, 10.0, step=0.5)
    min_coverage = 0.25

    params = {
        "booster": "gbtree",
        "tree_method": "hist",
        "max_depth": trial.suggest_int("max_depth", 2, 7),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 30.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "subsample": trial.suggest_float("subsample", 0.3, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "learning_rate": trial.suggest_float("learning_rate", 0.0075, 0.2, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 50.0, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 225, 450, step=25),
        "early_stopping_rounds": 200,
        "eval_metric": "rmse",
        "random_state": 16,
        "n_jobs": -1,
    }

    fold_scores = []
    fold_coverages = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr = y.iloc[tr_idx].to_numpy(dtype=float)
        y_va = y.iloc[va_idx].to_numpy(dtype=float)

        model = XGBRegressor(**params)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )

        y_pred_total = _predict_best(model, X_va)
        line_va = pd.to_numeric(X_va[BET365_LINE_COL], errors="coerce").to_numpy(dtype=float)

        score, coverage = ou_accuracy_with_threshold_totals(
            true_total=y_va,
            pred_total=np.asarray(y_pred_total, dtype=float),
            betting_line=line_va,
            threshold_abs=threshold_abs,
            min_coverage=min_coverage,
        )

        fold_scores.append(score)
        fold_coverages.append(coverage)

        trial.report(float(np.mean(fold_scores)), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    trial.set_user_attr("mean_coverage", float(np.mean(fold_coverages)))
    return float(np.mean(fold_scores))

# ----------------------------
# Run the search (2 to 3 hours)
# ----------------------------
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=16),
    pruner=MedianPruner(n_warmup_steps=2),
)

study.optimize(lambda t: objective(t, X_train, y_train), timeout=2 * 3600, n_jobs=1)

print("Best value (CV OU success):", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(k, v)

# ----------------------------
# Train final model on full train set
# ----------------------------
best_params = study.best_params.copy()
best_threshold = best_params.pop("threshold_abs")  # strategy threshold

final_params = {
    "booster": "gbtree",
    "tree_method": "hist",
    "random_state": 16,
    "n_jobs": -1,
    **best_params,
}

final_model = XGBRegressor(**final_params)



[I 2026-02-27 13:50:15,067] A new study created in memory with name: no-name-88775fd2-c2e5-457b-94f2-6945eaa2ec85
[I 2026-02-27 13:51:34,141] Trial 0 finished with value: 0.5514496759042569 and parameters: {'threshold_abs': 2.0, 'max_depth': 5, 'min_child_weight': 0.8219695696031055, 'gamma': 0.22800975066403162, 'subsample': 0.5525101847434408, 'colsample_bytree': 0.5338485650147733, 'learning_rate': 0.071971944311379, 'reg_alpha': 2.975656697106555e-07, 'reg_lambda': 3.478796625225186e-06, 'n_estimators': 450}. Best is trial 0 with value: 0.5514496759042569.
[I 2026-02-27 13:52:16,127] Trial 1 finished with value: 0.0 and parameters: {'threshold_abs': 5.5, 'max_depth': 2, 'min_child_weight': 3.2561820715318097, 'gamma': 0.7922608676158321, 'subsample': 0.4751969147208164, 'colsample_bytree': 0.5760923536613185, 'learning_rate': 0.07385949950651723, 'reg_alpha': 0.0001507914760359121, 'reg_lambda': 4.526442350789216e-05, 'n_estimators': 325}. Best is trial 0 with value: 0.551449675904

Best value (CV OU success): 0.5998996277085791
Best params:
threshold_abs 3.5
max_depth 5
min_child_weight 12.710399242652588
gamma 1.4032136090720946
subsample 0.38837129497210277
colsample_bytree 0.969708245947691
learning_rate 0.06639730039353646
reg_alpha 1.2944697996495302e-06
reg_lambda 0.00025930298465685854
n_estimators 375


## Check in Test set

In [27]:
final_model.fit(X_train, y_train, verbose=False)
results_df_final, y_pred_test = evaluate_ou_thresholds(
    model=final_model,
    X_test=X_test,
    y_test_total=y_test,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),  # 0..10
)

# Pretty display
display(results_df_final.style.format({
    "pct_of_test": "{:.1%}",
    "ou_accuracy": "{:.2%}",
}))



,threshold_abs_pred_vs_line_gt,n_games,pct_of_test,ou_accuracy
0,0,1095,100.0%,53.94%
1,1,964,88.0%,54.16%
2,2,837,76.4%,54.30%
3,3,700,63.9%,54.48%
4,4,578,52.8%,56.12%
5,5,481,43.9%,56.93%
6,6,389,35.5%,56.62%
7,7,328,30.0%,58.46%
8,8,255,23.3%,57.14%
9,9,193,17.6%,56.02%


In [28]:
feature_importances = final_model.feature_importances_
important_features = np.argsort(feature_importances)[::-1]  
feature_importances = pd.DataFrame({
    'Feature': X_train.columns[important_features],
    'Importance': feature_importances[important_features]
}).sort_values(by="Importance", ascending=False)
feature_importances

,Feature,Importance
0,total_fanduel_line_over,0.016590
1,odds_implied_points_max,0.009568
2,odds_total_line_books_mean,0.006692
3,total_draftkings_line_over,0.006255
4,total_caesars_line_over,0.003720
...,...,...
939,odds_ml_away_prob_novig_fanduel,0.000000
940,TOP5_INJURED_STREAK_TS_PCT_BEFORE_TEAM_HOME,0.000000
941,REF_TRIO_TOTAL_POINTS_STD_BEFORE,0.000000
942,IS_OVER_LINE_LAST_HOME_AWAY_10_MATCHES_BEFORE_...,0.000000


In [29]:

import json
from pathlib import Path

import joblib
import numpy as np
from xgboost import XGBRegressor


def train_final_model(
    X_all,
    y_all,
    study,
    *,
    use_early_stopping: bool = True,
    val_frac: float = 0.1,
    random_state: int = 16,
):
    """
    Train a final XGBRegressor using Optuna best params.
    Also returns the betting threshold (threshold_abs) stored in the study params.
    """
    best_params = study.best_params.copy()
    best_threshold = float(best_params.pop("threshold_abs"))

    final_params = {
        "booster": "gbtree",
        "tree_method": "hist",
        "random_state": random_state,
        "n_jobs": -1,
        "eval_metric": "rmse",
        **best_params,
    }

    model = XGBRegressor(**final_params)

    if not use_early_stopping:
        # Train on all data without early stopping
        model.fit(X_all, y_all, verbose=False)
        return model, best_threshold

    # Keep early stopping by holding out a small validation split
    n = len(X_all)
    rng = np.random.default_rng(random_state)
    idx = np.arange(n)
    rng.shuffle(idx)

    n_val = max(1, int(round(val_frac * n)))
    val_idx = idx[:n_val]
    tr_idx = idx[n_val:]

    X_tr, y_tr = X_all.iloc[tr_idx], y_all.iloc[tr_idx]
    X_va, y_va = X_all.iloc[val_idx], y_all.iloc[val_idx]

    model.set_params(early_stopping_rounds=200)
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False,
    )
    return model, best_threshold


def save_model_bundle(
    model: XGBRegressor,
    threshold_abs: float,
    feature_names: list[str],
    out_dir: str | Path,
    name: str = "xgb_total_points_ou_model",
):
    """
    Saves:
      - model.joblib (sklearn wrapper)
      - metadata.json (threshold + feature order)
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    model_path = out_dir / f"{name}.joblib"
    meta_path = out_dir / f"{name}.meta.json"

    joblib.dump(model, model_path)

    metadata = {
        "threshold_abs": float(threshold_abs),
        "feature_names": list(feature_names),
        "betting_line_column": BET365_LINE_COL,
        "target": "TOTAL_POINTS",
        "xgboost_version_note": "Saved via joblib XGBRegressor wrapper",
    }
    meta_path.write_text(json.dumps(metadata, indent=2))

    return model_path, meta_path


def load_model_bundle(model_path: str | Path, meta_path: str | Path):
    model = joblib.load(model_path)
    metadata = json.loads(Path(meta_path).read_text())
    return model, metadata


# -------------------------
# Example usage
# -------------------------

# 1) Build ALL data
X_all = X
y_all = y

# 2) Train final model
final_model, best_threshold = train_final_model(
    X_all=X_all,
    y_all=y_all,
    study=study,
    use_early_stopping=True,
    val_frac=0.1,
    random_state=16,
)

# 3) Save model + metadata (threshold + feature ordering)
model_path, meta_path = save_model_bundle(
    model=final_model,
    threshold_abs=best_threshold,
    feature_names=list(X_all.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/",
    name="drop_na_cols_5_all_data_xgb_total_points_27_02_26",
)

print("Saved:", model_path)
print("Saved:", meta_path)

# 4) Load later
loaded_model, meta = load_model_bundle(model_path, meta_path)
print("Loaded threshold_abs:", meta["threshold_abs"])


Saved: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/drop_na_cols_5_all_data_xgb_total_points_27_02_26.joblib
Saved: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/drop_na_cols_5_all_data_xgb_total_points_27_02_26.meta.json
Loaded threshold_abs: 3.5
